# 💼 03. 실전 — 정적 사이트 크롤링
## — 사람인 채용공고 완전 공략

> **이 노트북에서 배울 것**
> 1. robots.txt 확인 자동화
> 2. 사람인 채용공고 수집 (1페이지 → 여러 페이지)
> 3. 여러 키워드 한번에 수집
> 4. 데이터 정제 (문자열 → 숫자)
> 5. CSV 저장 & 간단한 분석

> ⚠️ **VS Code 로컬 환경 필수!** Colab에서는 IP 차단으로 안 됩니다.


## 1. 사람인 robots.txt 확인

크롤링 전 항상 첫 번째로 하는 것!


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 표준 헤더 — 매번 이걸 복사해서 시작
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# robots.txt 확인
response = requests.get('https://m.bunjang.co.kr/',
                 headers=HEADERS, timeout=10)
response.encoding = 'utf-8'                
print(response.text[:600])


<!doctype html>
<html lang="ko-KR">
  <head>
    <script id="bgzt-plugin-init">window.__BGZT_DOMAIN__='bunjang.co.kr';</script>

    <meta charset="UTF-8" />
    <meta name="robots" content="noimageindex, noarchive" />

    <meta
      name="viewport"
      content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no, viewport-fit=cover"
    />

    <title>번개장터</title>

    <link rel="icon" type="image/svg+xml" href="https://static.bunjang.co.kr/web/ui/favicon.ico" />

    <meta
      name="keywords"
      content="번개장터,번장,마켓,중고나라,Market,중고카페,C2C,연예인용품,스타굿즈,오픈마켓,저렴,개인간 거


## 2. 사람인 페이지 구조 파악

**개발자도구로 확인한 HTML 구조:**
```html
<div class="item_recruit">          ← 채용공고 하나
  <strong class="corp_name">
    <a>회사명</a>
  </strong>
  <h2 class="job_tit">
    <a>공고 제목</a>
  </h2>
  <div class="job_condition">
    <span>지역</span>
    <span>경력</span>
    <span>학력</span>
    <span>고용형태</span>
  </div>
  <div class="job_date">
    <span class="date">마감일</span>
  </div>
</div>
```


In [ ]:
# 사람인 1페이지 수집

keyword = '데이터분석'
url = f'https://www.saramin.co.kr/zf_user/search/recruit?searchType=search&searchword={keyword}'

response = requests.get(url, headers=HEADERS, timeout=15)
response.encoding = 'utf-8'
print(f"상태 코드: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')

# 채용공고 카드 찾기
jobs = soup.select('div.item_recruit')
print(f"발견된 공고: {len(jobs)}건")


In [ ]:
# 첫 번째 공고 구조 확인
if jobs:
    first_job = jobs[0]
    print("=== 첫 번째 공고 HTML ===")
    print(first_job.prettify()[:800])


In [ ]:
# 전체 데이터 추출

rows = []

for job in jobs:
    # 회사명
    company_el = job.select_one('strong.corp_name a')
    company = company_el.text.strip() if company_el else None

    # 공고 제목
    title_el = job.select_one('h2.job_tit a')
    title = title_el.text.strip() if title_el else None

    # 조건들 (지역, 경력, 학력, 고용형태)
    conditions = job.select('div.job_condition span')
    cond_texts = [c.text.strip() for c in conditions]

    # 마감일
    deadline_el = job.select_one('div.job_date span.date')
    deadline = deadline_el.text.strip() if deadline_el else None

    rows.append({
        '회사명': company,
        '공고제목': title,
        '지역': cond_texts[0] if len(cond_texts) > 0 else None,
        '경력': cond_texts[1] if len(cond_texts) > 1 else None,
        '학력': cond_texts[2] if len(cond_texts) > 2 else None,
        '고용형태': cond_texts[3] if len(cond_texts) > 3 else None,
        '마감일': deadline,
    })

df = pd.DataFrame(rows)
print(f"수집 완료: {len(df)}건")
df[['회사명', '공고제목', '지역', '경력']].head(10)


## 3. 여러 페이지 수집 (페이지네이션)

사람인 URL 패턴:
```
1페이지: ?searchword=키워드&recruitPage=1
2페이지: ?searchword=키워드&recruitPage=2
3페이지: ?searchword=키워드&recruitPage=3
```

→ URL의 숫자만 바꾸면 됩니다!


In [ ]:
# 여러 페이지 수집

def crawl_saramin(keyword, max_page=3):
    """사람인에서 키워드 검색 결과를 max_page 페이지까지 수집"""
    all_jobs = []

    for page in range(1, max_page + 1):
        url = (f'https://www.saramin.co.kr/zf_user/search/recruit'
               f'?searchType=search&searchword={keyword}&recruitPage={page}')

        response = requests.get(url, headers=HEADERS, timeout=15)
        response.encoding = 'utf-8'

        if response.status_code != 200:
            print(f"  {page}페이지 오류: {response.status_code}")
            break

        soup = BeautifulSoup(response.text, 'html.parser')
        jobs = soup.select('div.item_recruit')

        for job in jobs:
            company_el = job.select_one('strong.corp_name a')
            title_el = job.select_one('h2.job_tit a')
            conditions = job.select('div.job_condition span')
            cond_texts = [c.text.strip() for c in conditions]
            deadline_el = job.select_one('div.job_date span.date')

            all_jobs.append({
                '키워드': keyword,
                '페이지': page,
                '회사명': company_el.text.strip() if company_el else None,
                '공고제목': title_el.text.strip() if title_el else None,
                '지역': cond_texts[0] if len(cond_texts) > 0 else None,
                '경력': cond_texts[1] if len(cond_texts) > 1 else None,
                '학력': cond_texts[2] if len(cond_texts) > 2 else None,
                '마감일': deadline_el.text.strip() if deadline_el else None,
            })

        print(f"  {page}페이지 완료: {len(jobs)}건")
        time.sleep(1)  # 서버 부담 줄이기!

    return pd.DataFrame(all_jobs)


# 실행
print("=== 데이터분석 채용공고 수집 ===")
df_data = crawl_saramin('데이터분석', max_page=3)
print(f"\n총 {len(df_data)}건 수집 완료!")


In [ ]:
# 여러 키워드 한번에 수집

keywords = ['데이터분석', '파이썬', 'SQL']
frames = []

for kw in keywords:
    print(f"\n[{kw}] 수집 중...")
    df_kw = crawl_saramin(kw, max_page=2)
    frames.append(df_kw)
    time.sleep(2)  # 키워드 사이 2초 대기

df_all = pd.concat(frames, ignore_index=True)
print(f"\n전체 수집: {len(df_all)}건")
print(df_all.groupby('키워드').size().rename('공고수'))


## 4. 데이터 분석 & 정제

수집한 데이터를 바로 분석해봅시다!


In [ ]:
# 지역별 공고 수
print("=== 지역별 공고 수 (상위 10) ===")
print(df_all['지역'].value_counts().head(10).to_string())

print("\n=== 경력 조건별 공고 수 ===")
print(df_all['경력'].value_counts().head(10).to_string())


In [ ]:
# 서울 강남구 공고만 필터링
gangnam = df_all[df_all['지역'].str.contains('강남', na=False)]
print(f"강남 지역 공고: {len(gangnam)}건")
print(gangnam[['키워드', '회사명', '공고제목', '경력']].head(10).to_string(index=False))


In [ ]:
# 경력무관 or 신입 공고만 필터링
entry = df_all[
    df_all['경력'].isin(['경력무관', '신입', '신입·경력'])
]
print(f"신입/경력무관 공고: {len(entry)}건")
print(entry[['키워드', '회사명', '공고제목', '지역']].head(10).to_string(index=False))


In [ ]:
# CSV 저장

save_path = r'사람인_채용공고.csv'
# 윈도우에서는 r 문자열 또는 / 사용 권장

df_all.to_csv(save_path, index=False, encoding='utf-8-sig')
# utf-8-sig: 엑셀에서 열었을 때 한글 안 깨지게!

print(f"저장 완료: {save_path}")
print(f"파일 크기: {len(df_all)}행 × {len(df_all.columns)}열")
print(f"컬럼: {list(df_all.columns)}")


## 5. 취업 포트폴리오 아이디어

이 데이터로 만들 수 있는 분석:

```
📊 Tableau / Power BI 시각화:
├── 지역 히트맵: 어느 지역에 공고가 많나?
├── 키워드별 경력 요구사항 비교
├── 마감일 타임라인
└── 회사별 채용 규모

🐍 Python 추가 분석:
├── 공고 제목에서 기술 스택 추출 (Python, SQL, R 등)
├── 경력 연차 분포
└── 월별 채용 트렌드
```


In [ ]:
# 보너스: 공고 제목에서 기술 스택 추출

tech_keywords = ['Python', 'SQL', 'R', 'Tableau', 'Excel',
                 '머신러닝', '딥러닝', 'AI', 'AWS', 'Spark']

for tech in tech_keywords:
    count = df_all['공고제목'].str.contains(tech, case=False, na=False).sum()
    if count > 0:
        print(f"  {tech:15}: {count}건")


## ✅ 오늘 배운 것 정리

```python
# 완성된 크롤링 템플릿

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}

# 1. robots.txt 확인
# 2. 페이지 가져오기
# 3. BeautifulSoup 파싱
# 4. 데이터 추출 (select + if else None)
# 5. time.sleep(1) — 항상!
# 6. CSV 저장 (encoding='utf-8-sig')
```

> **다음 노트북**: JavaScript 렌더링 사이트는 Selenium으로!
